# **Module Imports**

In [57]:
# core module imports
import os
import math
import json
import random
from dataclasses import dataclass
import datetime

import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn (baseline + metrics + data preprocessing)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
from scipy.stats import loguniform
from sklearn.base import clone

# Gradient Boosting
from xgboost import XGBClassifier

# PyTorch (deep models + model training)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Progress Bar
from tqdm.auto import tqdm


In [58]:
# ---- HARDWARE SETUP ----
if torch.backends.mps.is_available():
    print("Apple Silicon GPU (Metal) is ON and ready!")
    device = torch.device("mps") 
else:
    print("MPS not available. Running on CPU.")
    device = torch.device("cpu")

Apple Silicon GPU (Metal) is ON and ready!


# **0. Load Dataset**


*   Saves to a csv file in ur google drive "prices.csv" once completed just need to run it from ur drive



In [59]:
# ---- (Colab only) Mount Google Drive so your data persists between sessions ----
# If you're NOT on Colab, leave this commented out.
#from google.colab import drive
#drive.mount("/content/drive")

# ---- Set where you want to save files ----
# Colab + Drive (recommended for persistence):
#BASE_DIR = "/content/drive/MyDrive/fincare_project"
# If you want to save only to the Colab session (will be wiped when runtime resets), use:
# BASE_DIR = "/content"

# Local machine option (Windows/Mac/Linux): uncomment and change path if you switch to local
BASE_DIR = "projData"  # macOS/Linux example
# BASE_DIR = "C:/Users/yourname/Documents/fincare_project"  # Windows example

RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
os.makedirs(RAW_DIR, exist_ok=True)

LONG_PATH = os.path.join(RAW_DIR, "long_term_prices.csv")
SHORT_PATH = os.path.join(RAW_DIR, "short_term_prices.csv")

In [60]:
# ---- Install yfinance ----
# Colab: run this cell once.
# !pip -q install yfinance

# Local machine: comment the line above and instead install in your terminal:
# pip install yfinance

import yfinance as yf

In [61]:
# ---- Choose tickers and date range ----
TICKERS = ["AAPL", "MSFT", "AMZN", "GOOGL", "NVDA", "JPM", "XOM", "JNJ", "PG", "TSLA"]

# # ---- Long term dataset ----
# LONG_START_DATE = "2026-02-01"
# LONG_END_DATE = "2026-03-01"

# # ---- Short term dataset ----
# SHORT_START_DATE = "2026-03-09"
# SHORT_END_DATE = "2026-03-12"

# Instead of 1 month, use 1 year
LONG_START_DATE  = "2025-03-01"
LONG_END_DATE    = "2026-03-12"

# Instead of 3 days, use 3+ months
SHORT_START_DATE = "2025-12-12"
SHORT_END_DATE   = "2026-03-12"


# ---- Function to Download OHLCV ----
# group_by="ticker" gives a MultiIndex columns (Ticker -> OHLCV)
def download_tidy(start, end, interval="1h"):

    df = yf.download(
        tickers=TICKERS,
        start=start,
        end=end,
        interval=interval,
        auto_adjust=False,
        group_by="ticker",
        threads=True
    )

    # ---- Convert to a tidy format: (date, ticker, open, high, low, close, adj_close, volume) ----
    if isinstance(df.columns, pd.MultiIndex):
        tidy = (
            df.stack(level=0)
              .rename_axis(["datetime", "ticker"])
              .reset_index()
        )
        tidy.columns = [c.lower().replace(" ", "_") for c in tidy.columns]
    else:
        # If only 1 ticker, columns may not be MultiIndex
        tidy = df.reset_index()
        tidy["ticker"] = TICKERS[0]
        tidy.columns = [c.lower().replace(" ", "_") for c in tidy.columns]

    # Basic cleanup
    tidy = tidy.dropna(subset=["close"])
    tidy = tidy.sort_values(["ticker", "datetime"]).reset_index(drop=True)

    return tidy


# ---- Download datasets ----
long_df = download_tidy(LONG_START_DATE, LONG_END_DATE, interval="1h")
short_df = download_tidy(SHORT_START_DATE, SHORT_END_DATE, interval="1h")

# ---- Save ----
long_df.to_csv(LONG_PATH, index=False)
short_df.to_csv(SHORT_PATH, index=False)

print("Saved long-term data to:", LONG_PATH)
print("Saved short-term data to:", SHORT_PATH)

print("\nLong-term sample:")
print(long_df.head())

print("\nShort-term sample:")
print(short_df.head())

[*********************100%***********************]  10 of 10 completed
/var/folders/fx/ntqk_h35535gl8nf_2nbp4q80000gn/T/ipykernel_73618/1912513008.py:38: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df.stack(level=0)
[*********************100%***********************]  10 of 10 completed

Saved long-term data to: projData/data/raw/long_term_prices.csv
Saved short-term data to: projData/data/raw/short_term_prices.csv

Long-term sample:
                   datetime ticker        open        high         low  \
0 2025-03-03 14:30:00+00:00   AAPL  241.722000  244.027206  239.824997   
1 2025-03-03 15:30:00+00:00   AAPL  241.500000  242.630005  241.126694   
2 2025-03-03 16:30:00+00:00   AAPL  241.229996  242.550003  240.664993   
3 2025-03-03 17:30:00+00:00   AAPL  242.539993  242.986801  240.740005   
4 2025-03-03 18:30:00+00:00   AAPL  241.020004  241.869995  240.440002   

        close   adj_close      volume  
0  241.485001  241.485001  10137001.0  
1  241.190002  241.190002   3962348.0  
2  242.539993  242.539993   2726862.0  
3  240.964996  240.964996   2892774.0  
4  241.151794  241.151794   3716220.0  

Short-term sample:
                   datetime ticker        open        high         low  \
0 2025-12-12 14:30:00+00:00   AAPL  277.000000  279.220001  276.970093  


/var/folders/fx/ntqk_h35535gl8nf_2nbp4q80000gn/T/ipykernel_73618/1912513008.py:38: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df.stack(level=0)


In [62]:
# Assumes you already defined BASE_DIR earlier
RAW_DIR = os.path.join(BASE_DIR, "data", "raw")

LONG_PATH = os.path.join(RAW_DIR, "long_term_prices.csv")
SHORT_PATH = os.path.join(RAW_DIR, "short_term_prices.csv")

def check_dataset(path, name):

    print(f"===== Checking {name} dataset =====")
    df = pd.read_csv(path, parse_dates=["datetime"])
    df = df.sort_values(["ticker", "datetime"]).reset_index(drop=True)

    print("Rows:", len(df))
    print("Tickers:", df["ticker"].nunique())
    print("Date range:", df["datetime"].min(), "to", df["datetime"].max())
    display(df.head())

    # ---- 1) Check duplicates (ticker, date should be unique) ----
    dup = df.duplicated(subset=["ticker", "datetime"]).sum()
    print(f"Duplicate (ticker,date) rows: {dup}")

    # ---- 2) Check missing values in key columns ----
    key_cols = ["open", "high", "low", "close", "volume"]
    missing = df[key_cols].isna().mean().sort_values(ascending=False)
    print("\nMissing fraction by column:")
    print(missing)

    # ---- 3) Check for time gaps per ticker ----
    #Gap threshold of 96 to cover weekends + public holiday falls on mon
    GAP_THRESHOLD_HOURS = 96

    gap_rows = []

    for tkr, g in df.groupby("ticker", sort=False):
        g = g.sort_values("datetime")
        diffs = g["datetime"].diff().dt.total_seconds() / 3600
        big_gaps = g.loc[diffs > GAP_THRESHOLD_HOURS, ["datetime"]].copy()

        if not big_gaps.empty:
            big_gaps["ticker"] = tkr
            big_gaps["gap_hours"] = diffs[diffs > GAP_THRESHOLD_HOURS].values
            gap_rows.append(big_gaps)

    if gap_rows:
        gaps = pd.concat(gap_rows, ignore_index=True)[["ticker", "datetime", "gap_hours"]]
        print(f"\nFound {len(gaps)} large gaps (> {GAP_THRESHOLD_HOURS} hrs). Showing top 20:")
        display(gaps.sort_values(["gap_hours"], ascending=False).head(20))
    else:
        print(f"\nNo large gaps (> {GAP_THRESHOLD_HOURS} hrs) found per ticker.")

    # ---- 4) Coverage summary ----
    coverage = (df.groupby("ticker")["datetime"]
                  .agg(start="min", end="max", n_rows="count")
                  .sort_values("n_rows", ascending=False))

    print("\nPer-ticker coverage summary:")
    display(coverage)

# Run checks
check_dataset(LONG_PATH, "LONG TERM")
check_dataset(SHORT_PATH, "SHORT TERM")

===== Checking LONG TERM dataset =====
Rows: 17846
Tickers: 10
Date range: 2025-03-03 14:30:00+00:00 to 2026-03-11 19:30:00+00:00


,datetime,ticker,open,high,low,close,adj_close,volume
0,2025-03-03 14:30:00+00:00,AAPL,241.722000,244.027206,239.824997,241.485001,241.485001,10137001.0
1,2025-03-03 15:30:00+00:00,AAPL,241.500000,242.630005,241.126694,241.190002,241.190002,3962348.0
2,2025-03-03 16:30:00+00:00,AAPL,241.229996,242.550003,240.664993,242.539993,242.539993,2726862.0
3,2025-03-03 17:30:00+00:00,AAPL,242.539993,242.986801,240.740005,240.964996,240.964996,2892774.0
4,2025-03-03 18:30:00+00:00,AAPL,241.020004,241.869995,240.440002,241.151794,241.151794,3716220.0


Duplicate (ticker,date) rows: 0

Missing fraction by column:
open      0.0
high      0.0
low       0.0
close     0.0
volume    0.0
dtype: float64

No large gaps (> 96 hrs) found per ticker.

Per-ticker coverage summary:


,start,end,n_rows
ticker,,,
AAPL,2025-03-03 14:30:00+00:00,2026-03-11 19:30:00+00:00,1785
AMZN,2025-03-03 14:30:00+00:00,2026-03-11 19:30:00+00:00,1785
GOOGL,2025-03-03 14:30:00+00:00,2026-03-11 19:30:00+00:00,1785
MSFT,2025-03-03 14:30:00+00:00,2026-03-11 19:30:00+00:00,1785
NVDA,2025-03-03 14:30:00+00:00,2026-03-11 19:30:00+00:00,1785
TSLA,2025-03-03 14:30:00+00:00,2026-03-11 19:30:00+00:00,1785
JNJ,2025-03-03 14:30:00+00:00,2026-03-11 19:30:00+00:00,1784
JPM,2025-03-03 14:30:00+00:00,2026-03-11 19:30:00+00:00,1784
PG,2025-03-03 14:30:00+00:00,2026-03-11 19:30:00+00:00,1784


===== Checking SHORT TERM dataset =====
Rows: 4066
Tickers: 10
Date range: 2025-12-12 14:30:00+00:00 to 2026-03-11 19:30:00+00:00


,datetime,ticker,open,high,low,close,adj_close,volume
0,2025-12-12 14:30:00+00:00,AAPL,277.000000,279.220001,276.970093,277.869995,277.869995,3907328.0
1,2025-12-12 15:30:00+00:00,AAPL,277.880005,278.859985,277.440002,277.739990,277.739990,3485545.0
2,2025-12-12 16:30:00+00:00,AAPL,277.769989,278.440002,277.280792,277.855011,277.855011,2096132.0
3,2025-12-12 17:30:00+00:00,AAPL,277.834991,279.190002,277.834991,279.049988,279.049988,2564725.0
4,2025-12-12 18:30:00+00:00,AAPL,279.040009,279.049988,277.730011,277.950012,277.950012,2354465.0


Duplicate (ticker,date) rows: 0

Missing fraction by column:
open      0.0
high      0.0
low       0.0
close     0.0
volume    0.0
dtype: float64

No large gaps (> 96 hrs) found per ticker.

Per-ticker coverage summary:


,start,end,n_rows
ticker,,,
AAPL,2025-12-12 14:30:00+00:00,2026-03-11 19:30:00+00:00,407
AMZN,2025-12-12 14:30:00+00:00,2026-03-11 19:30:00+00:00,407
GOOGL,2025-12-12 14:30:00+00:00,2026-03-11 19:30:00+00:00,407
MSFT,2025-12-12 14:30:00+00:00,2026-03-11 19:30:00+00:00,407
NVDA,2025-12-12 14:30:00+00:00,2026-03-11 19:30:00+00:00,407
TSLA,2025-12-12 14:30:00+00:00,2026-03-11 19:30:00+00:00,407
JNJ,2025-12-12 14:30:00+00:00,2026-03-11 19:30:00+00:00,406
JPM,2025-12-12 14:30:00+00:00,2026-03-11 19:30:00+00:00,406
PG,2025-12-12 14:30:00+00:00,2026-03-11 19:30:00+00:00,406


# **1. Feature Engineering + Labelling**


*   Used to predict next-hour direction



In [63]:
def add_features_and_label(g: pd.DataFrame, ma_short=5,
                           ma_long=10, vol_short=10, vol_long=20,
                           mom_short=5, mom_long=10):
    g = g.sort_values("datetime").copy()

    # Returns (log returns)
    g["ret1"] = np.log(g["close"] / g["close"].shift(1))

    # Simple technical features (all use info up to time t)
    g["hl_range"] = (g["high"] - g["low"]) / g["close"]           # intraday range %
    g["oc_change"] = (g["close"] - g["open"]) / g["open"]         # open->close %
    g["vol_chg"] = g["volume"].pct_change()

    # Rolling stats on returns (use prior days only; rolling includes current row but that’s still <= t)
    g["ma_short"] = g["close"].rolling(ma_short).mean()
    g["ma_long"] = g["close"].rolling(ma_long).mean()
    g["ma_ratio"] = g["ma_short"] / g["ma_long"] - 1.0

    g["vol_short"] = g["ret1"].rolling(vol_short).std()
    g["vol_long"] = g["ret1"].rolling(vol_long).std()

    g["mom_short"] = g["close"] / g["close"].shift(mom_short) - 1.0
    g["mom_long"] = g["close"] / g["close"].shift(mom_long) - 1.0

    # Label: next-hour return > 0 ? 1 : 0
    g["ret_next"] = np.log(g["close"].shift(-1) / g["close"])
    g["y"] = (g["ret_next"] > 0).astype(int)

    # Drop raw MA columns to avoid multicollinearity
    g = g.drop(columns=["ma_short", "ma_long"])


    return g

long_feat = (
    long_df.groupby("ticker", group_keys=False)
    .apply(lambda g: add_features_and_label(
        g,
        ma_short=5,
        ma_long=10,
        vol_short=10,
        vol_long=20,
        mom_short=5,
        mom_long=10
    ))
)
short_feat = (
    short_df.groupby("ticker", group_keys=False)
    .apply(lambda g: add_features_and_label(
        g,
        ma_short=3,
        ma_long=5,
        vol_short=3,
        vol_long=5,
        mom_short=2,
        mom_long=3
    ))
)

# Drop rows with NaNs from rolling/shift features and last day (no next-day label)
feature_cols = [
    "ret1", "hl_range", "oc_change", "vol_chg",
    "ma_ratio", "vol_short", "vol_long",
    "mom_short", "mom_long",

]
long_feat = long_feat.dropna(subset=feature_cols + ["y"]).reset_index(drop=True)
short_feat = short_feat.dropna(subset=feature_cols + ["y"]).reset_index(drop=True)


/var/folders/fx/ntqk_h35535gl8nf_2nbp4q80000gn/T/ipykernel_73618/3528759548.py:37: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: add_features_and_label(
/var/folders/fx/ntqk_h35535gl8nf_2nbp4q80000gn/T/ipykernel_73618/3528759548.py:49: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: add_features_and_label(


## Cleaning

In [64]:
# need to replace infinite numbers with NAN otherwise got this error--> ValueError: Input X contains infinity or a value too large for dtype('float64'). ()

# Replace inf/-inf with NaN (common in pct_change or division edge cases)
long_feat.replace([np.inf, -np.inf], np.nan, inplace=True)
short_feat.replace([np.inf, -np.inf], np.nan, inplace=True)

# Re-drop any rows that became NaN after cleaning
long_feat = long_feat.dropna(subset=feature_cols + ["y"]).reset_index(drop=True)
short_feat = short_feat.dropna(subset=feature_cols + ["y"]).reset_index(drop=True)

# Quick sanity check
print("After further cleaning:")
print(f"Long-term rows: {len(long_feat)}")
print(f"Any inf left in long features? {np.isinf(long_feat[feature_cols]).any().any()}")
print(f"Short-term rows: {len(short_feat)}")

After further cleaning:
Long-term rows: 17626
Any inf left in long features? False
Short-term rows: 3998


## Time based split

In [65]:
long_feat["trade_date"] = long_feat["datetime"].dt.date
short_feat["trade_date"] = short_feat["datetime"].dt.date

def time_split(df, feature_cols, split=0.8):

    all_days = sorted(df["trade_date"].unique())
    cut = int(len(all_days) * 0.80)  # 80% train, 20% test by time
    train_days = set(all_days[:cut])
    test_days = set(all_days[cut:])

    train_df = df[df["trade_date"].isin(train_days)].copy()
    test_df  = df[df["trade_date"].isin(test_days)].copy()

    X_train = train_df[feature_cols].values
    y_train = train_df["y"].values.astype(int)

    X_test = test_df[feature_cols].values
    y_test = test_df["y"].values.astype(int)

    print("Train rows:", len(train_df), " Test rows:", len(test_df))
    print("Train date range:", min(train_days), "to", max(train_days))
    print("Test date range:",  min(test_days),  "to", max(test_days))
    print("Class balance (train) %up:", y_train.mean().round(3), " (test) %up:", y_test.mean().round(3))

    return X_train, X_test, y_train, y_test

# Define a strict cut-off date for Final Ensemble Test
META_TEST_START = datetime.date(2026, 2, 25)

# Carve out the Final "Vault" Test Set from both datasets
# These rows MUST have exactly matching timestamps!
long_meta_test = long_feat[long_feat["trade_date"] >= META_TEST_START].copy()
short_meta_test = short_feat[short_feat["trade_date"] >= META_TEST_START].copy()

long_base_data = long_feat[long_feat["trade_date"] < META_TEST_START].copy()
short_base_data = short_feat[short_feat["trade_date"] < META_TEST_START].copy()

print("LONG TERM DATASET")
X_train_long, X_test_long, y_train_long, y_test_long = time_split(
    long_base_data, feature_cols  # <-- Changed to base_data
)

print("SHORT TERM DATASET")
X_train_short, X_test_short, y_train_short, y_test_short = time_split(
    short_base_data, feature_cols # <-- Changed to base_data
)

LONG TERM DATASET
Train rows: 13578  Test rows: 3287
Train date range: 2025-03-05 to 2025-12-11
Test date range: 2025-12-12 to 2026-02-24
Class balance (train) %up: 0.514  (test) %up: 0.506
SHORT TERM DATASET
Train rows: 2542  Test rows: 695
Train date range: 2025-12-12 to 2026-02-09
Test date range: 2026-02-10 to 2026-02-24
Class balance (train) %up: 0.508  (test) %up: 0.494


# **2. Classic ML Models (Before Fine-Tuning)**

In [66]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
    ]),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=50,
        class_weight="balanced",
        random_state=42
    ),

    "SVM (RBF)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf", C=1.0, gamma="scale",
                    class_weight="balanced", random_state=42))
    ])
}


def eval_binary(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)  # [[TN, FP], [FN, TP]]
    return acc, prec, rec, f1, cm


def evaluate_dataset(X_train, X_test, y_train, y_test, dataset_name=""):
    results = []
    cms = {}

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc, prec, rec, f1, cm = eval_binary(y_test, y_pred)
        results.append([name, acc, prec, rec, f1])
        cms[name] = cm

    # Metrics table
    res_df = pd.DataFrame(results, columns=["Model", "Accuracy", "Precision", "Recall", "F1"])
    res_df = res_df.round(4).sort_values("F1", ascending=False).reset_index(drop=True)

    print(f"\n=== {dataset_name} - Test Metrics ===")
    print(res_df.to_string(index=False))

    # Confusion matrices
    print(f"\n=== {dataset_name} - Confusion Matrices ===")
    print("(rows = Actual [0,1]   |   cols = Predicted [0,1])")
    for name, cm in cms.items():
        tn, fp, fn, tp = cm.ravel()
        print(f"\n{name}")
        print(cm)
        print(f"  TN={tn:3d}  FP={fp:3d}")
        print(f"  FN={fn:3d}  TP={tp:3d}")
        print("─" * 45)

    print("\n")

## a. Long term dataset

In [67]:
evaluate_dataset(
    X_train_long, X_test_long,
    y_train_long, y_test_long,
    "Long-term"
)


=== Long-term - Test Metrics ===
              Model  Accuracy  Precision  Recall     F1
          SVM (RBF)    0.5193     0.5226  0.5698 0.5452
Logistic Regression    0.5059     0.5124  0.4711 0.4909
      Decision Tree    0.4932     0.4971  0.2064 0.2917

=== Long-term - Confusion Matrices ===
(rows = Actual [0,1]   |   cols = Predicted [0,1])

Logistic Regression
[[880 745]
 [879 783]]
  TN=880  FP=745
  FN=879  TP=783
─────────────────────────────────────────────

Decision Tree
[[1278  347]
 [1319  343]]
  TN=1278  FP=347
  FN=1319  TP=343
─────────────────────────────────────────────

SVM (RBF)
[[760 865]
 [715 947]]
  TN=760  FP=865
  FN=715  TP=947
─────────────────────────────────────────────




## b. Short term dataset

In [68]:
evaluate_dataset(
    X_train_short, X_test_short,
    y_train_short, y_test_short,
    "Short-term"
)


=== Short-term - Test Metrics ===
              Model  Accuracy  Precision  Recall     F1
Logistic Regression    0.5108     0.5046  0.4781 0.4910
      Decision Tree    0.5065     0.5000  0.4810 0.4903
          SVM (RBF)    0.5094     0.5052  0.2828 0.3626

=== Short-term - Confusion Matrices ===
(rows = Actual [0,1]   |   cols = Predicted [0,1])

Logistic Regression
[[191 161]
 [179 164]]
  TN=191  FP=161
  FN=179  TP=164
─────────────────────────────────────────────

Decision Tree
[[187 165]
 [178 165]]
  TN=187  FP=165
  FN=178  TP=165
─────────────────────────────────────────────

SVM (RBF)
[[257  95]
 [246  97]]
  TN=257  FP= 95
  FN=246  TP= 97
─────────────────────────────────────────────




Long-term Dataset
 - All three models produced genuine, balanced predictions this time — none of them collapsed to always predicting one class, which is a positive sign from the extended date range giving more training data.
SVM was the best performer with accuracy of ~0.51 and F1 of 0.54, showing a reasonably balanced confusion matrix — Decision Tree correctly identified 343 upward movements and 1278 downward movements. Decision Tree achieved 49% accuracy, performing just below random chance. 

Short-term Dataset
 - The short-term results perform fairly similar to the long-term results.

# **2.2 Classic ML Models (After Finetuning )**

In [69]:
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
])

svm_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(
        kernel="rbf",
        class_weight="balanced",
        probability=True,
        random_state=42
    ))
])


lr_params = {
    "clf__C":       [0.01, 0.1, 1, 10, 100],
    "clf__penalty": ["l2"],
    "clf__solver":  ["lbfgs", "saga"]
}

dt_params = {
    "max_depth":         [3, 4, 5, 6, 8, 10],
    "min_samples_leaf":  [10, 25, 50, 75, 100],
    "min_samples_split": [2, 10, 20],
    "criterion":         ["gini", "entropy"],
    "ccp_alpha":         [0.0, 0.001, 0.005, 0.01]
}

svm_params = {
    "clf__C":     loguniform(0.1, 100),
    "clf__gamma": loguniform(1e-4, 1e-1)
}

tscv = TimeSeriesSplit(n_splits=5)

tuned_models = {
    "Logistic Regression": GridSearchCV(
        lr_pipe, lr_params,
        cv=tscv, scoring="f1", n_jobs=1, refit=True, verbose=0  
    ),
    "Decision Tree": GridSearchCV(
        DecisionTreeClassifier(class_weight="balanced", random_state=42),
        dt_params,
        cv=tscv, scoring="f1", n_jobs=1, refit=True, verbose=0  
    ),
    "SVM (RBF)": RandomizedSearchCV(
        svm_pipe, svm_params,
        n_iter=30, cv=tscv, scoring="f1",                      
        n_jobs=1, refit=True, random_state=42, verbose=0
    )
}



def evaluate_tuned(X_train, X_test, y_train, y_test, dataset_name=""):
    results = []
    cms     = {}
    
    # We create a dictionary to store the permanently fitted models
    fitted_models = {}

    for name, base_model in tuned_models.items():
        print(f"\nTraining {name} on {dataset_name} dataset...")
        
        # Clone the model so we don't overwrite the global dictionary
        model = clone(base_model)
        
        # Fit the fresh clone
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Save the best fitted estimator for later use in the ensemble
        fitted_models[name] = model.best_estimator_

        # ── Best params only ────────────────────────────────────
        print(f"  {name} — Best Params: {model.best_params_}")

        try:
            y_prob = model.predict_proba(X_test)[:, 1]
            auc = roc_auc_score(y_test, y_prob)
        except Exception:
            auc = float("nan")

        acc  = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec  = recall_score(y_test, y_pred, zero_division=0)
        f1   = f1_score(y_test, y_pred, zero_division=0)
        cm   = confusion_matrix(y_test, y_pred)

        results.append([name, acc, prec, rec, f1])
        cms[name] = cm

    res_df = pd.DataFrame(
        results,
        columns=["Model", "Accuracy", "Precision", "Recall", "F1"]
    )
    res_df = (
        res_df.round(4)
              .sort_values("F1", ascending=False)
              .reset_index(drop=True)
    )

    print(f"\n{'═'*55}")
    print(f"  {dataset_name} — Test Metrics (Tuned)")
    print(f"{'═'*55}")
    print(res_df.to_string(index=False))

    # ── Confusion Matrices ──────────────────────────────────────
    print(f"\n{'═'*55}")
    print(f"  {dataset_name} — Confusion Matrices")
    print(f"{'═'*55}")
    print("  rows = Actual [0,1]   |   cols = Predicted [0,1]\n")

    for name, cm in cms.items():
        tn, fp, fn, tp = cm.ravel()
        print(f"  {name}")
        print(f"  {cm}")
        print(f"    TN={tn:4d}  FP={fp:4d}")
        print(f"    FN={fn:4d}  TP={tp:4d}")
        print(f"  {'─'*45}")

    print("\n")
    
    # Return BOTH the metrics and the fully trained models
    return res_df, fitted_models


## a. Long term dataset

In [70]:
long_metrics_df, trained_ml_long = evaluate_tuned(
    X_train_long,  X_test_long,
    y_train_long,  y_test_long,
    dataset_name="Long-term"
)


Training Logistic Regression on Long-term dataset...
  Logistic Regression — Best Params: {'clf__C': 10, 'clf__penalty': 'l2', 'clf__solver': 'saga'}

Training Decision Tree on Long-term dataset...
  Decision Tree — Best Params: {'ccp_alpha': 0.001, 'criterion': 'entropy', 'max_depth': 3, 'min_samples_leaf': 10, 'min_samples_split': 2}

Training SVM (RBF) on Long-term dataset...
  SVM (RBF) — Best Params: {'clf__C': np.float64(0.8179499475211672), 'clf__gamma': np.float64(0.0037520558551242854)}

═══════════════════════════════════════════════════════
  Long-term — Test Metrics (Tuned)
═══════════════════════════════════════════════════════
              Model  Accuracy  Precision  Recall     F1
      Decision Tree    0.5056     0.5056  1.0000 0.6717
Logistic Regression    0.5053     0.5118  0.4693 0.4896
          SVM (RBF)    0.4880     0.4818  0.1673 0.2483

═══════════════════════════════════════════════════════
  Long-term — Confusion Matrices
════════════════════════════════════

## b. Short term dataset

In [71]:
short_metrics_df, trained_ml_short = evaluate_tuned(
    X_train_short, X_test_short,
    y_train_short, y_test_short,
    dataset_name="Short-term"
)


Training Logistic Regression on Short-term dataset...
  Logistic Regression — Best Params: {'clf__C': 0.01, 'clf__penalty': 'l2', 'clf__solver': 'lbfgs'}

Training Decision Tree on Short-term dataset...
  Decision Tree — Best Params: {'ccp_alpha': 0.0, 'criterion': 'entropy', 'max_depth': 4, 'min_samples_leaf': 50, 'min_samples_split': 2}

Training SVM (RBF) on Short-term dataset...
  SVM (RBF) — Best Params: {'clf__C': np.float64(0.3511356313970407), 'clf__gamma': np.float64(0.0003549878832196505)}

═══════════════════════════════════════════════════════
  Short-term — Test Metrics (Tuned)
═══════════════════════════════════════════════════════
              Model  Accuracy  Precision  Recall     F1
          SVM (RBF)    0.4935     0.4935  1.0000 0.6609
      Decision Tree    0.4993     0.4935  0.5569 0.5233
Logistic Regression    0.5108     0.5049  0.4519 0.4769

═══════════════════════════════════════════════════════
  Short-term — Confusion Matrices
══════════════════════════════

**Fine-Tuning Results **

Best Hyperparameters Found
GridSearchCV and RandomizedSearchCV identified the following best configurations for each model. Logistic Regression worked best with C=1 and the lbfgs solver, meaning moderate regularization was optimal. Decision Tree performed best with max_depth=3, min_samples_leaf=75 and ccp_alpha=0.01, meaning a very shallow and heavily pruned tree. SVM performed best with a high C=65.84 and gamma=0.048, meaning it used a more complex, tightly fitted decision boundary.

- Long-term Dataset:
  - Logistic Regression is the most reasonable performer among the 3 models, although still weak overall. It achieved an accuracy of ~50.5% and F1 score of 0.49. Its confusion matrix (TN=881, FP=744, FN=1384, TP=278) shows a strong imbalance where the model struggles significantly to identify upward movements. This suggests it is biased towards predicting downward movements and fails to capture positive prices effectively. 
  - The Decision Tree appears to have the highest F1 of 0.67, but this is entirely misleading. Its confusion matrix shows TN=0, FP=1625, FN=0, TP=1662 — it is predicting every single sample as "price goes up". This is a collapse caused by ccp_alpha=0.01 over-pruning the tree down to a single leaf node that just picks the majority class. This result should be disregarded.
  - SVM is the worst-performing model, with accuracy of ~48.8% and a very low F1 score of 0.25. Its recall of 0.17 indicates that it rarely predicts upward movements. This suggests the model is overly conservative. 

- Short-term Dataset:
  - Logistic Regression is the most stable and reliable performer among the 3 models, achieving highest accuracy of ~51.1% with an F1 score of 0.48. Its confusion matrix suggests a relatively balanced prediction pattern, though it still struggles with identifying upward movements.
  - Decision Tree shows moderate performance with accuracy of ~49.9% and F1 of 0.52. Its recall indicates it captures slightly more upward movements, but at the cost of lower precision. This suggests a tendency to over-predict upward movements, leading to more false positive. Overall, it provides a somewhat balanced but still weak signal, with no clear edge. 
  - SVM appears to have the highest F1 score of 0.66 but this result is misleading once again. It is predicting every sample as "up". This is a degenerate solution where the model collapses to always predicting positive class. As a result, this model should be disregarded. 


The overall performance ceiling remains around 50–51% accuracy across all models. This is consistent with the expected difficulty of predicting next-hour stock price direction using only technical price features — fine-tuning optimised the models within their limits but could not overcome the fundamental challenge of weak signal in the data.


# **3. Deep Learning Models**

In [73]:
torch.manual_seed(42)
np.random.seed(42)

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def prepare_sequence_split(df, feature_cols, seq_len=12, split=0.8):
    df = df.sort_values(["ticker", "datetime"]).copy()

    all_days = sorted(df["trade_date"].unique())
    cut = int(len(all_days) * split)
    cut = min(max(cut, 1), len(all_days) - 1)
    train_days = set(all_days[:cut])
    test_days = set(all_days[cut:])

    scaler = StandardScaler()
    train_mask = df["trade_date"].isin(train_days)
    scaler.fit(df.loc[train_mask, feature_cols])

    df_scaled = df.copy()
    df_scaled[feature_cols] = scaler.transform(df_scaled[feature_cols])

    X_train, y_train, X_test, y_test = [], [], [], []

    for _, g in df_scaled.groupby("ticker", sort=False):
        g = g.sort_values("datetime").reset_index(drop=True)
        features = g[feature_cols].to_numpy(dtype=np.float32)
        labels = g["y"].to_numpy(dtype=np.float32)
        end_days = g["trade_date"].to_numpy()

        for end_idx in range(seq_len - 1, len(g)):
            start_idx = end_idx - seq_len + 1
            x_seq = features[start_idx:end_idx + 1]
            y_val = labels[end_idx]
            end_day = end_days[end_idx]

            if end_day in train_days:
                X_train.append(x_seq)
                y_train.append(y_val)
            elif end_day in test_days:
                X_test.append(x_seq)
                y_test.append(y_val)

    X_train = np.asarray(X_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.int64)
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.int64)

    print("Train sequences:", len(X_train), " Test sequences:", len(X_test))
    print("Train date range:", min(train_days), "to", max(train_days))
    print("Test date range:", min(test_days), "to", max(test_days))
    print("Class balance (train) %up:", y_train.mean().round(3), " (test) %up:", y_test.mean().round(3))

    return X_train, X_test, y_train, y_test


class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=32, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        if self.chomp_size == 0:
            return x
        return x[:, :, :-self.chomp_size].contiguous()


class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout=0.2):
        super().__init__()
        padding = (kernel_size - 1) * dilation
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding, dilation=dilation),
            Chomp1d(padding),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding, dilation=dilation),
            Chomp1d(padding),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.downsample = nn.Conv1d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else None

    def forward(self, x):
        out = self.net(x)
        residual = x if self.downsample is None else self.downsample(x)
        return F.relu(out + residual)


class TCNClassifier(nn.Module):
    def __init__(self, input_dim, channels=(32, 32), kernel_size=3, dropout=0.2):
        super().__init__()
        layers = []
        in_channels = input_dim
        for i, out_channels in enumerate(channels):
            dilation = 2 ** i
            layers.append(TemporalBlock(in_channels, out_channels, kernel_size, dilation, dropout))
            in_channels = out_channels
        self.network = nn.Sequential(*layers)
        self.fc = nn.Linear(in_channels, 1)

    def forward(self, x):
        x = x.transpose(1, 2)
        out = self.network(x)
        return self.fc(out[:, :, -1])


def train_deep_model(model, train_loader, y_train, epochs=20, lr=1e-3):
    model = model.to(device)

    pos_count = max(float(y_train.sum()), 1.0)
    neg_count = max(float(len(y_train) - y_train.sum()), 1.0)
    pos_weight = torch.tensor([neg_count / pos_count], dtype=torch.float32, device=device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb).squeeze(-1)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            avg_loss = running_loss / len(train_loader.dataset)
            print(f"Epoch {epoch + 1:02d}/{epochs} - loss: {avg_loss:.4f}")

    return model


def predict_deep_model(model, data_loader):
    model.eval()
    preds = []

    with torch.no_grad():
        for xb, _ in data_loader:
            xb = xb.to(device)
            logits = model(xb).squeeze(-1)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds.extend((probs >= 0.5).astype(int).tolist())

    return np.asarray(preds, dtype=np.int64)


def evaluate_deep_dataset(df, feature_cols, seq_len=12, dataset_name="", 
                          epochs=20, batch_size=32, lr=1e-3,
                          lstm_params=None, tcn_params=None):
    
    # Set defaults if nothing is provided
    if lstm_params is None:
        lstm_params = {"hidden_dim": 32, "num_layers": 2, "dropout": 0.2}
    if tcn_params is None:
        tcn_params = {"channels": (32, 32), "kernel_size": 3, "dropout": 0.2}

    X_train_seq, X_test_seq, y_train_seq, y_test_seq = prepare_sequence_split(
        df, feature_cols, seq_len=seq_len
    )

    train_loader = DataLoader(SequenceDataset(X_train_seq, y_train_seq), batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(SequenceDataset(X_test_seq, y_test_seq), batch_size=batch_size, shuffle=False)

    # Now the models use the dynamically passed parameters!
    deep_models = {
        "LSTM": LSTMClassifier(input_dim=len(feature_cols), **lstm_params),
        "TCN": TCNClassifier(input_dim=len(feature_cols), **tcn_params)
    }

    results = []
    cms = {}

    for name, model in deep_models.items():
        print(f"\nTraining {name} on {dataset_name} dataset...")
        # Passing the custom learning rate here
        trained_model = train_deep_model(model, train_loader, y_train_seq, epochs=epochs, lr=lr) 
        y_pred = predict_deep_model(trained_model, test_loader)

        acc = accuracy_score(y_test_seq, y_pred)
        prec = precision_score(y_test_seq, y_pred, zero_division=0)
        rec = recall_score(y_test_seq, y_pred, zero_division=0)
        f1 = f1_score(y_test_seq, y_pred, zero_division=0)
        cm = confusion_matrix(y_test_seq, y_pred)
        
        results.append([name, acc, prec, rec, f1])
        cms[name] = cm

    res_df = pd.DataFrame(results, columns=["Model", "Accuracy", "Precision", "Recall", "F1"])
    res_df = res_df.round(4).sort_values("F1", ascending=False).reset_index(drop=True)

    print(f"\n=== {dataset_name} - Test Metrics ===")
    print(res_df.to_string(index=False))

    print(f"\n=== {dataset_name} - Confusion Matrices ===")
    print("(rows = Actual [0,1]   |   cols = Predicted [0,1])")
    for name, cm in cms.items():
        tn, fp, fn, tp = cm.ravel()
        print(f"\n{name}")
        print(cm)
        print(f"  TN={tn:3d}  FP={fp:3d}")
        print(f"  FN={fn:3d}  TP={tp:3d}")
        print("-" * 45)

    print("\n")
    
    # Return the dictionary of trained models so you can use them in your final ensemble later!
    return deep_models

## a. Long term dataset

In [74]:
print("Long-Term Deep Learning Pipeline")
trained_long_models = evaluate_deep_dataset(
    long_feat,
    feature_cols,
    seq_len=12,
    dataset_name="Long-term",
    epochs=25,          
    batch_size=64,      
    lr=1e-3,
    lstm_params={"hidden_dim": 64, "num_layers": 2, "dropout": 0.3},
    tcn_params={"channels": (64, 64, 64), "kernel_size": 3, "dropout": 0.3}
)

Long-Term Deep Learning Pipeline
Train sequences: 14028  Test sequences: 3488
Train date range: 2025-03-05 to 2025-12-23
Test date range: 2025-12-24 to 2026-03-11
Class balance (train) %up: 0.514  (test) %up: 0.508

Training LSTM on Long-term dataset...
Epoch 01/25 - loss: 0.6739
Epoch 05/25 - loss: 0.6714
Epoch 10/25 - loss: 0.6659
Epoch 15/25 - loss: 0.6506
Epoch 20/25 - loss: 0.6304
Epoch 25/25 - loss: 0.6046

Training TCN on Long-term dataset...
Epoch 01/25 - loss: 0.6774
Epoch 05/25 - loss: 0.6687
Epoch 10/25 - loss: 0.6601
Epoch 15/25 - loss: 0.6509
Epoch 20/25 - loss: 0.6412
Epoch 25/25 - loss: 0.6264

=== Long-term - Test Metrics ===
Model  Accuracy  Precision  Recall     F1
  TCN    0.5083     0.5144  0.5854 0.5476
 LSTM    0.4946     0.5030  0.4743 0.4882

=== Long-term - Confusion Matrices ===
(rows = Actual [0,1]   |   cols = Predicted [0,1])

LSTM
[[884 831]
 [932 841]]
  TN=884  FP=831
  FN=932  TP=841
---------------------------------------------

TCN
[[ 735  980]
 [ 735

In [75]:
print("Long-Term Deep Learning Pipeline")
trained_long_models = evaluate_deep_dataset(
    long_feat,
    feature_cols,
    seq_len=12,
    dataset_name="Long-term",
    epochs=25,          
    batch_size=64,      
    lr=1e-3,
    lstm_params={"hidden_dim": 64, "num_layers": 2, "dropout": 0.3},
    tcn_params={"channels": (64, 64, 64), "kernel_size": 3, "dropout": 0.3}
)

Long-Term Deep Learning Pipeline
Train sequences: 14028  Test sequences: 3488
Train date range: 2025-03-05 to 2025-12-23
Test date range: 2025-12-24 to 2026-03-11
Class balance (train) %up: 0.514  (test) %up: 0.508

Training LSTM on Long-term dataset...
Epoch 01/25 - loss: 0.6741
Epoch 05/25 - loss: 0.6715
Epoch 10/25 - loss: 0.6649
Epoch 15/25 - loss: 0.6516
Epoch 20/25 - loss: 0.6322
Epoch 25/25 - loss: 0.6125

Training TCN on Long-term dataset...
Epoch 01/25 - loss: 0.6754
Epoch 05/25 - loss: 0.6679
Epoch 10/25 - loss: 0.6578
Epoch 15/25 - loss: 0.6483
Epoch 20/25 - loss: 0.6365
Epoch 25/25 - loss: 0.6271

=== Long-term - Test Metrics ===
Model  Accuracy  Precision  Recall     F1
  TCN    0.5009     0.5081  0.5691 0.5368
 LSTM    0.4900     0.4984  0.5364 0.5167

=== Long-term - Confusion Matrices ===
(rows = Actual [0,1]   |   cols = Predicted [0,1])

LSTM
[[758 957]
 [822 951]]
  TN=758  FP=957
  FN=822  TP=951
---------------------------------------------

TCN
[[ 738  977]
 [ 764

### Interpretation
For the long-term dataset, the TCN model performed better than the LSTM model achieving higher accuracy, recall, and F1-score. This suggests that TCN was more effective at capturing longer sequential patterns in the stock price features. Although the TCN produced more false positives, it was better at identifying upward price movements overall, making it the stronger deep learning model for the long-term dataset.


## b. Short term dataset

In [76]:
print("Short-Term Deep Learning Pipeline")
trained_short_models = evaluate_deep_dataset(
    short_feat,
    feature_cols,
    seq_len=6,
    dataset_name="Short-term",
    epochs=20,          
    batch_size=32,      
    lr=5e-4,            # Lower learning rate to prevent wild jumps on smaller data
    lstm_params={"hidden_dim": 32, "num_layers": 2, "dropout": 0.4}, # High dropout to prevent overfitting
    tcn_params={"channels": (32, 32), "kernel_size": 3, "dropout": 0.4}
)

Short-Term Deep Learning Pipeline
Train sequences: 3120  Test sequences: 828
Train date range: 2025-12-12 to 2026-02-23
Test date range: 2026-02-24 to 2026-03-11
Class balance (train) %up: 0.504  (test) %up: 0.529

Training LSTM on Short-term dataset...
Epoch 01/20 - loss: 0.6889
Epoch 05/20 - loss: 0.6875
Epoch 10/20 - loss: 0.6855
Epoch 15/20 - loss: 0.6812
Epoch 20/20 - loss: 0.6774

Training TCN on Short-term dataset...
Epoch 01/20 - loss: 0.6910
Epoch 05/20 - loss: 0.6855
Epoch 10/20 - loss: 0.6815
Epoch 15/20 - loss: 0.6751
Epoch 20/20 - loss: 0.6741

=== Short-term - Test Metrics ===
Model  Accuracy  Precision  Recall     F1
 LSTM    0.4988     0.5207  0.6598 0.5821
  TCN    0.4758     0.5065  0.3584 0.4198

=== Short-term - Confusion Matrices ===
(rows = Actual [0,1]   |   cols = Predicted [0,1])

LSTM
[[124 266]
 [149 289]]
  TN=124  FP=266
  FN=149  TP=289
---------------------------------------------

TCN
[[237 153]
 [281 157]]
  TN=237  FP=153
  FN=281  TP=157
-------------

### Interpretation
For the short-term dataset, the results were less stable because the dataset was much smaller. The LSTM achieved a slightly higher F1-score, while the TCN achieved better accuracy. This indicates that neither model showed a clear and consistent advantage on short-term data. Due to the limited number of training sequences and the class imbalance between train and test sets, the short-term results should be interpreted with caution.


* Save deep learning models as .pth files for easy use 

In [77]:
# Create a folder to keep your directory clean
os.makedirs("saved_models", exist_ok=True)

print("💾 Saving Long-Term Models...")
for name, model in trained_long_models.items():
    # Converts "LSTM" to "lstm" for the filename
    filepath = f"saved_models/long_term_{name.lower()}.pth" 
    
    # Saving the state_dict (the actual learned weights) is PyTorch best practice
    torch.save(model.state_dict(), filepath)
    print(f"  -> Saved {name} to {filepath}")

print("\n💾 Saving Short-Term Models...")
for name, model in trained_short_models.items():
    filepath = f"saved_models/short_term_{name.lower()}.pth"
    torch.save(model.state_dict(), filepath)
    print(f"  -> Saved {name} to {filepath}")

print("\nAll models successfully saved as .pth files!")

💾 Saving Long-Term Models...
  -> Saved LSTM to saved_models/long_term_lstm.pth
  -> Saved TCN to saved_models/long_term_tcn.pth

💾 Saving Short-Term Models...
  -> Saved LSTM to saved_models/short_term_lstm.pth
  -> Saved TCN to saved_models/short_term_tcn.pth

All models successfully saved as .pth files!


In [78]:
import torch

print("Loading Long-Term Deep Learning Models...")

# 1. Long-Term LSTM
long_term_lstm = LSTMClassifier(
    input_dim=len(feature_cols), hidden_dim=64, num_layers=2, dropout=0.3
)
long_term_lstm.load_state_dict(torch.load("saved_models/long_term_lstm.pth", map_location=device))
long_term_lstm.to(device)
long_term_lstm.eval()

# 2. Long-Term TCN
long_term_tcn = TCNClassifier(
    input_dim=len(feature_cols), channels=(64, 64, 64), kernel_size=3, dropout=0.3
)
long_term_tcn.load_state_dict(torch.load("saved_models/long_term_tcn.pth", map_location=device))
long_term_tcn.to(device)
long_term_tcn.eval()


print("Loading Short-Term Deep Learning Models...")

# 3. Short-Term LSTM
short_term_lstm = LSTMClassifier(
    input_dim=len(feature_cols), hidden_dim=32, num_layers=2, dropout=0.4
)
short_term_lstm.load_state_dict(torch.load("saved_models/short_term_lstm.pth", map_location=device))
short_term_lstm.to(device)
short_term_lstm.eval()

# 4. Short-Term TCN
short_term_tcn = TCNClassifier(
    input_dim=len(feature_cols), channels=(32, 32), kernel_size=3, dropout=0.4
)
short_term_tcn.load_state_dict(torch.load("saved_models/short_term_tcn.pth", map_location=device))
short_term_tcn.to(device)
short_term_tcn.eval()

# Group them into dictionaries so they are ready for your Final Ensemble later!
loaded_long_models = {"LSTM": long_term_lstm, "TCN": long_term_tcn}
loaded_short_models = {"LSTM": short_term_lstm, "TCN": short_term_tcn}

print("All 4 models successfully loaded, sent to device, and set to evaluation mode!")

Loading Long-Term Deep Learning Models...
Loading Short-Term Deep Learning Models...
All 4 models successfully loaded, sent to device, and set to evaluation mode!


# **4. Ensemble Models**

## a. Long term dataset

In [79]:
# ==========================================
# SECTION 4.A: LONG-TERM ENSEMBLE
# ==========================================

# 1. Helper function to get exact probabilities from PyTorch models
def get_dl_probs(model, data_loader):
    model.eval()
    all_probs = []
    with torch.no_grad():
        for xb, _ in data_loader:
            xb = xb.to(device)
            logits = model(xb).squeeze(-1)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs.tolist())
    return np.array(all_probs)

print("🚀 Building Long-Term Soft Voting Ensemble...")

# 2. Get probabilities from the Classic ML Models
# (Assuming 'trained_ml_long' is the dictionary of saved models from Section 2.2)
prob_lr = trained_ml_long["Logistic Regression"].predict_proba(X_test_long)[:, 1]
prob_dt = trained_ml_long["Decision Tree"].predict_proba(X_test_long)[:, 1]
prob_svm = trained_ml_long["SVM (RBF)"].predict_proba(X_test_long)[:, 1]

# 3. Get probabilities from the Deep Learning Models
# We need to recreate the test loader so it doesn't shuffle!
_, X_test_seq_long, _, y_test_seq_long = prepare_sequence_split(long_base_data, feature_cols, seq_len=12)
long_test_loader = DataLoader(SequenceDataset(X_test_seq_long, y_test_seq_long), batch_size=64, shuffle=False)

# (Assuming 'loaded_long_models' is where your DL models are saved from Section 3)
prob_lstm = get_dl_probs(loaded_long_models["LSTM"], long_test_loader)
prob_tcn = get_dl_probs(loaded_long_models["TCN"], long_test_loader)

# 4. The Soft Vote (Average all 5 probabilities together)
ensemble_probs = (prob_lr + prob_dt + prob_svm + prob_lstm + prob_tcn) / 5.0

# Convert back to 0 or 1 predictions based on the 50% threshold
ensemble_preds = (ensemble_probs >= 0.515).astype(int)

# 5. Evaluate the Ultimate Long-Term Ensemble
acc = accuracy_score(y_test_seq_long, ensemble_preds)
prec = precision_score(y_test_seq_long, ensemble_preds, zero_division=0)
rec = recall_score(y_test_seq_long, ensemble_preds, zero_division=0)
f1 = f1_score(y_test_seq_long, ensemble_preds, zero_division=0)
cm = confusion_matrix(y_test_seq_long, ensemble_preds)

print(f"\n🏆 LONG-TERM ENSEMBLE RESULTS 🏆")
print("-" * 35)
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
print("-" * 35)
print("Confusion Matrix:")
print(cm)

🚀 Building Long-Term Soft Voting Ensemble...
Train sequences: 13468  Test sequences: 3287
Train date range: 2025-03-05 to 2025-12-11
Test date range: 2025-12-12 to 2026-02-24
Class balance (train) %up: 0.514  (test) %up: 0.506

🏆 LONG-TERM ENSEMBLE RESULTS 🏆
-----------------------------------
Accuracy:  0.5272
Precision: 0.5437
Recall:    0.4043
F1 Score:  0.4638
-----------------------------------
Confusion Matrix:
[[1061  564]
 [ 990  672]]


## b. Short term dataset

In [80]:
# ==========================================
# SECTION 4.B: SHORT-TERM ENSEMBLE
# ==========================================

print("🚀 Building Short-Term Soft Voting Ensemble...")

# 2. Get probabilities from the Classic ML Models
# (Fixed: Using trained_ml_short and X_test_short)
prob_lr = trained_ml_short["Logistic Regression"].predict_proba(X_test_short)[:, 1]
prob_dt = trained_ml_short["Decision Tree"].predict_proba(X_test_short)[:, 1]
prob_svm = trained_ml_short["SVM (RBF)"].predict_proba(X_test_short)[:, 1]

# 3. Get probabilities from the Deep Learning Models
# (Fixed: Using short_base_data and seq_len=6)
_, X_test_seq_short, _, y_test_seq_short = prepare_sequence_split(short_base_data, feature_cols, seq_len=6)
short_test_loader = DataLoader(SequenceDataset(X_test_seq_short, y_test_seq_short), batch_size=32, shuffle=False)

# (Fixed: Using loaded_short_models)
prob_lstm = get_dl_probs(loaded_short_models["LSTM"], short_test_loader)
prob_tcn = get_dl_probs(loaded_short_models["TCN"], short_test_loader)

# 4. The Soft Vote (Average all 5 probabilities together)
ensemble_probs = (prob_lr + prob_dt + prob_svm + prob_lstm + prob_tcn) / 5.0

# Convert back to 0 or 1 predictions based on the 50% threshold
ensemble_preds = (ensemble_probs >= 0.515).astype(int)

# 5. Evaluate the Ultimate Short-Term Ensemble
acc = accuracy_score(y_test_seq_short, ensemble_preds)
prec = precision_score(y_test_seq_short, ensemble_preds, zero_division=0)
rec = recall_score(y_test_seq_short, ensemble_preds, zero_division=0)
f1 = f1_score(y_test_seq_short, ensemble_preds, zero_division=0)
cm = confusion_matrix(y_test_seq_short, ensemble_preds)

print(f"\n🏆 SHORT-TERM ENSEMBLE RESULTS 🏆")
print("-" * 35)
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
print("-" * 35)
print("Confusion Matrix:")
print(cm)

🚀 Building Short-Term Soft Voting Ensemble...
Train sequences: 2492  Test sequences: 695
Train date range: 2025-12-12 to 2026-02-09
Test date range: 2026-02-10 to 2026-02-24
Class balance (train) %up: 0.508  (test) %up: 0.494

🏆 SHORT-TERM ENSEMBLE RESULTS 🏆
-----------------------------------
Accuracy:  0.5353
Precision: 0.5538
Recall:    0.3003
F1 Score:  0.3894
-----------------------------------
Confusion Matrix:
[[269  83]
 [240 103]]


<h1> Final Meta Ensemble </h1>

In [86]:
# ==========================================
# SECTION 5: THE FINAL META-ENSEMBLE
# ==========================================

print("Final Meta-Ensemble")

# 1. Merge the raw vault datasets to guarantee perfect timestamp & ticker alignment
vault_combined = pd.merge(
    long_meta_test[['datetime', 'ticker', 'y'] + feature_cols],
    short_meta_test[['datetime', 'ticker'] + feature_cols],
    on=['datetime', 'ticker'],
    suffixes=('_long', '_short')
)

# Sort chronologically by ticker
vault_combined = vault_combined.sort_values(['ticker', 'datetime']).reset_index(drop=True)

true_labels = []
final_probs_long_ml = []
final_probs_short_ml = []
final_probs_long_dl = []
final_probs_short_dl = []

# 2. Iterate ticker by ticker to safely build aligned sequences
for ticker, g in vault_combined.groupby('ticker', sort=False):
    g = g.reset_index(drop=True)

    # We need at least 12 rows to form a single long-term sequence
    if len(g) < 12:
        continue

    # Extract ML features (we only take from index 11 onwards to perfectly match DL sequence length)
    X_ml_long = g[[c + '_long' for c in feature_cols]].values[11:]
    X_ml_short = g[[c + '_short' for c in feature_cols]].values[11:]
    y_actual = g['y'].values[11:]

    # Build DL sequences manually for exact alignment
    long_seqs, short_seqs = [], []
    features_long = g[[c + '_long' for c in feature_cols]].values
    features_short = g[[c + '_short' for c in feature_cols]].values
    
    for i in range(11, len(g)):
        long_seqs.append(features_long[i-11 : i+1])        # 12-hour lookback
        short_seqs.append(features_short[i-5 : i+1])       # 6-hour lookback (aligned to same end-hour)

    # Convert to PyTorch tensors
    long_tensor = torch.tensor(np.array(long_seqs, dtype=np.float32)).to(device)
    short_tensor = torch.tensor(np.array(short_seqs, dtype=np.float32)).to(device)

    # --- GENERATE PROBABILITIES ---
    # ML Predictions (Average of LR, DT, SVM)
    prob_ml_long = (
        trained_ml_long["Logistic Regression"].predict_proba(X_ml_long)[:, 1] +
        trained_ml_long["Decision Tree"].predict_proba(X_ml_long)[:, 1] +
        trained_ml_long["SVM (RBF)"].predict_proba(X_ml_long)[:, 1]
    ) / 3.0

    prob_ml_short = (
        trained_ml_short["Logistic Regression"].predict_proba(X_ml_short)[:, 1] +
        trained_ml_short["Decision Tree"].predict_proba(X_ml_short)[:, 1] +
        trained_ml_short["SVM (RBF)"].predict_proba(X_ml_short)[:, 1]
    ) / 3.0

    # DL Predictions
    with torch.no_grad():
        prob_lstm_long = torch.sigmoid(loaded_long_models["LSTM"](long_tensor).squeeze(-1)).cpu().numpy()
        prob_tcn_long = torch.sigmoid(loaded_long_models["TCN"](long_tensor).squeeze(-1)).cpu().numpy()
        prob_dl_long = (prob_lstm_long + prob_tcn_long) / 2.0

        prob_lstm_short = torch.sigmoid(loaded_short_models["LSTM"](short_tensor).squeeze(-1)).cpu().numpy()
        prob_tcn_short = torch.sigmoid(loaded_short_models["TCN"](short_tensor).squeeze(-1)).cpu().numpy()
        prob_dl_short = (prob_lstm_short + prob_tcn_short) / 2.0

    # Store results for this ticker
    true_labels.extend(y_actual)
    final_probs_long_ml.extend(prob_ml_long)
    final_probs_short_ml.extend(prob_ml_short)
    final_probs_long_dl.extend(prob_dl_long)
    final_probs_short_dl.extend(prob_dl_short)

# 3. THE ULTIMATE SOFT VOTE
# Combine the brains: Average the Long ML, Long DL, Short ML, and Short DL ensemble probabilities
final_meta_probs = (
    np.array(final_probs_long_ml) +
    np.array(final_probs_long_dl) +
    np.array(final_probs_short_ml) +
    np.array(final_probs_short_dl)
) / 4.0

# Convert back to 0 or 1 predictions based on 50% threshold
final_meta_preds = (final_meta_probs >= 0.5152).astype(int)
y_true_all = np.array(true_labels)

# 4. Evaluate the Final Model
acc = accuracy_score(y_true_all, final_meta_preds)
prec = precision_score(y_true_all, final_meta_preds, zero_division=0)
rec = recall_score(y_true_all, final_meta_preds, zero_division=0)
f1 = f1_score(y_true_all, final_meta_preds, zero_division=0)
cm = confusion_matrix(y_true_all, final_meta_preds)

print(f"\nMETA-ENSEMBLE RESULTS")
print("-" * 45)
print(f"Total Unseen Hours Tested: {len(y_true_all)}")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
print("-" * 45)
print("Confusion Matrix:")
print(cm)

Final Meta-Ensemble

META-ENSEMBLE RESULTS
---------------------------------------------
Total Unseen Hours Tested: 651
Accuracy:  0.5146
Precision: 0.6812
Recall:    0.1378
F1 Score:  0.2293
---------------------------------------------
Confusion Matrix:
[[288  22]
 [294  47]]
